<a href="https://colab.research.google.com/github/purnimamarasinghe2005-lgtm/BI-and-Data-Mining/blob/main/Final_Business_Intelligence_and_Data_Mining_1_4_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting Early Readmission of Diabetic Patients: A Business Intelligence Analysis of 130 US Hospitals (1999–2008)

**Module:** Business Intelligence and Data Mining (UFCFMM)  
**Assessment:** Task 2 – Group Project Portfolio  

**Group Members:**  
AL Wai Lwin Chan  
Sin Thier  
Sadeep Madushan Yati Kindage  
Prunima  

**Datasets:**  
[diabetic_data.csv](https://drive.google.com/file/d/19ZaXM0NpKTD1bQaxtgKqyZR28tH6wRcq/view?usp=drive_link)  
[IDS_mapping.csv](https://drive.google.com/file/d/1J3bVU-mJaJas84V4LqSuFhhBI21U9_NY/view?usp=drive_link)


This notebook studies whether hospital encounter data can help identify diabetic patients who are likely to be readmitted within 30 days. Readmission matters from a business perspective because it affects cost, bed capacity, and quality of care. The notebook follows the full business intelligence lifecycle required in the brief: problem definition, dataset justification, preprocessing, exploratory analysis, modelling, evaluation, recommendations, and ethical reflection. The dataset is associated with the study by Strack *et al.* (2014), which supports its academic relevance.


## [1] Background of Diabetic Readmission Problem
Hospital readmission within 30 days is a significant issue in healthcare systems, particularly for patients with chronic conditions such as diabetes. These patients are more likely to experience complications, poor disease management, and comorbidities, which increase the likelihood of repeated hospitalisations (Strack et al., 2014).

High readmission rates often indicate gaps in treatment effectiveness, discharge planning, or follow-up care (Zuckerman et al., 2016).

### Business Importance
From a business perspective, high readmission rates increase operational costs, reduce hospital capacity, and negatively impact performance metrics. Hospitals may face financial penalties from regulators and insurance providers, and readmission rates are often used as indicators of service quality and efficiency (Centers for Medicare & Medicaid Services, 2023).

### Project Objectives
This project aims to develop a predictive model to identify diabetic patients at risk of 30-day readmission. The objectives are to analyse key factors influencing readmission, apply machine learning techniques, generate actionable insights, and evaluate ethical and practical implications.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from plotly import graph_objects as go
!pip install squarify
import squarify

## [2] Dataset Description

### Source of Dataset
The dataset is obtained from the UCI Machine Learning Repository and contains over 100,000 hospital encounters from multiple US hospitals between 1999 and 2008.

### Key Attributes
The dataset includes:
- Demographic variables (age, gender, race)
- Clinical variables (diagnoses, lab results, medications)
- Hospital utilisation data (inpatient, outpatient, emergency visits)
- Administrative data (admission type, discharge details)

These attributes provide a comprehensive view of patient health and hospital interactions.

### Target Variable Definition
The original readmission variable contains three categories: `<30`, `>30`, and `NO`. This is transformed into a binary variable:
- 1: readmitted within 30 days  
- 0: not readmitted within 30 days  

This simplifies modelling and aligns with the objective of identifying high-risk patients.

In [ ]:
##2 LOAD DATASETS

df = pd.read_csv("diabetic_data.csv")
ids_map = pd.read_csv("IDS_mapping.csv")

##[3] DATA CLEANING & FEATURE ENGINEERING

### [3.1] Data Cleaning
Missing values represented by “?” are converted to NaN for consistency. Several features are removed due to irrelevance or high missingness:
- encounter_id and patient_nbr (identifiers)
- weight (high missing values)
- payer_code and medical_specialty (incomplete and less relevant)

These steps reduce noise and improve data quality.

In [ ]:
df.replace('?', np.nan, inplace=True)

df.drop([
    'encounter_id','patient_nbr','weight',
    'payer_code','medical_specialty'
], axis=1, inplace=True, errors='ignore')

# Only perform conversion and drop if 'readmitted' column exists
if 'readmitted' in df.columns:
    df['readmitted_binary'] = df['readmitted'].apply(lambda x: 1 if x == "<30" else 0)
    df.drop('readmitted', axis=1, inplace=True)

### [3.2] Feature Engineering

Key transformations include:

- Converting age groups into numerical values (age_num) to enable quantitative analysis  
- Binary encoding for variables such as medication change and diabetes medication usage to simplify categorical inputs  
- Ordinal encoding for clinical variables such as insulin dosage, A1C results, and glucose levels to preserve meaningful order  
- Grouping diagnosis codes into broader categories to reduce dimensionality while retaining clinical relevance  
- Creating a composite feature, *treatment_intensity*, by combining the number of procedures, lab tests, and medications to represent overall treatment complexity  

These transformations improve model performance by reducing noise, preserving meaningful relationships, and enabling algorithms to capture patterns more effectively.

In [ ]:
age_map = {
    "[0-10)":5,"[10-20)":15,"[20-30)":25,"[30-40)":35,"[40-50)":45,
    "[50-60)":55,"[60-70)":65,"[70-80)":75,"[80-90)":85,"[90-100)":95
}
df["age_num"] = df["age"].map(age_map)

df["change_bin"] = df["change"].replace({"Ch":1,"No":0})
df["diabetesMed_bin"] = df["diabetesMed"].replace({"Yes":1,"No":0})

A1C_map = {"Norm":0,">7":1,">8":2}
glu_map = {"Norm":0,">200":1,">300":2}
insulin_map = {"No":0,"Steady":1,"Up":2,"Down":3}

df["A1Cresult_bin"] = df["A1Cresult"].map(A1C_map)
df["max_glu_serum_bin"] = df["max_glu_serum"].map(glu_map)
df["insulin_bin"] = df["insulin"].map(insulin_map)

df["diag_1_group"] = df["diag_1"].astype(str).str[0]

df["treatment_intensity"] = (
    df["num_procedures"] +
    df["num_lab_procedures"] +
    df["num_medications"]
)

/tmp/ipykernel_34643/3556199546.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["change_bin"] = df["change"].replace({"Ch":1,"No":0})
/tmp/ipykernel_34643/3556199546.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["diabetesMed_bin"] = df["diabetesMed"].replace({"Yes":1,"No":0})


### [3.3] Final Dataset Overview
The final dataset consists of structured numerical and encoded categorical variables. Data types are consistent, and summary statistics confirm the distribution and scale of key features.

In [ ]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 53 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   race                      99493 non-null   object 
 1   gender                    101766 non-null  object 
 2   age                       101766 non-null  object 
 3   admission_type_id         101766 non-null  int64  
 4   discharge_disposition_id  101766 non-null  int64  
 5   admission_source_id       101766 non-null  int64  
 6   time_in_hospital          101766 non-null  int64  
 7   num_lab_procedures        101766 non-null  int64  
 8   num_procedures            101766 non-null  int64  
 9   num_medications           101766 non-null  int64  
 10  number_outpatient         101766 non-null  int64  
 11  number_emergency          101766 non-null  int64  
 12  number_inpatient          101766 non-null  int64  
 13  diag_1                    101745 non-null  o

,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,readmitted_binary,age_num,change_bin,diabetesMed_bin,A1Cresult_bin,max_glu_serum_bin,insulin_bin,treatment_intensity
count,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,17018.000000,5346.000000,101766.000000,101766.000000
mean,2.024006,3.715642,5.754437,4.395987,43.095641,1.339730,16.021844,0.369357,0.197836,0.635566,7.422607,0.111599,65.967022,0.461952,0.770031,1.189564,0.750655,0.885708,60.457216
std,1.445403,5.280166,4.064081,2.985108,19.674362,1.705807,8.127566,1.267265,0.930472,1.262863,1.933600,0.314874,15.940838,0.498553,0.420815,0.860297,0.812510,1.021758,23.588174
min,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000
25%,1.000000,1.000000,1.000000,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000,0.000000,55.000000,0.000000,1.000000,0.000000,0.000000,0.000000,46.000000
50%,1.000000,1.000000,7.000000,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000,0.000000,65.000000,0.000000,1.000000,1.000000,1.000000,1.000000,61.000000
75%,3.000000,4.000000,7.000000,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000,0.000000,75.000000,1.000000,1.000000,2.000000,1.000000,1.000000,76.000000
max,8.000000,28.000000,25.000000,14.000000,132.000000,6.000000,81.000000,42.000000,76.000000,21.000000,16.000000,1.000000,95.000000,1.000000,1.000000,2.000000,2.000000,3.000000,179.000000
